In [1]:
from langchain_community.document_loaders import TextLoader, CSVLoader, DirectoryLoader

#文档加载器
#txt文件加载
loader=TextLoader(
    file_path="../asset/load/01-langchain-utf-8.txt",
    encoding="utf-8",
)
docs=loader.load()
print(docs)

[Document(metadata={'source': '../asset/load/01-langchain-utf-8.txt'}, page_content='LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务')]


In [12]:
#加载CSv文件
from langchain_community.document_loaders import TextLoader, CSVLoader
loader=CSVLoader(file_path="../asset/load/02-load.csv",encoding="utf-8")
docs=loader.load()
print(docs)

[Document(metadata={'source': '../asset/load/02-load.csv', 'row': 0}, page_content='id: 1\ntitle: Introduction to Python\ncontent: Python is a popular programming language.\nauthor: John Doe'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 1}, page_content='id: 2\ntitle: Data Science Basics\ncontent: Data science involves statistics and machine learning.\nauthor: Jane Smith'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 2}, page_content='id: 3\ntitle: Web Development\ncontent: HTML, CSS and JavaScript are core web technologies.\nauthor: Mike Johnson'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 3}, page_content='id: 4\ntitle: Artificial Intelligence\ncontent: AI is transforming many industries.\nauthor: Sarah Williams')]


In [10]:
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
model=init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek"
)
from langchain_core.tools import tool


@tool
def introduce_self(
    name: str,
    sid: str,
    role: str,
    motto: str,
    hobbies: list[str],
    technologies: list[dict[str, str]],
) -> str:
    """根据用户提供的信息生成 Markdown 自我介绍。

    Args:
        name: 姓名。
        sid: 学号。
        role: 身份，例如人工智能专业学生。
        motto: 一句喜欢的座右铭。
        hobbies: 恰好 3 项技能或爱好。
        technologies: 恰好 3 门技术，每项包含 name、application、
            proficiency，分别表示技术名、应用场景、掌握程度。
    """
    if len(hobbies) != 3:
        return "请提供恰好 3 项技能或爱好。"

    if len(technologies) != 3:
        return "请提供恰好 3 门技术。"

    required_keys = {"name", "application", "proficiency"}
    if any(not required_keys.issubset(item) for item in technologies):
        return "每门技术都需要填写 name、application、proficiency。"

    hobby_text = "\n".join(f"- {item}" for item in hobbies)

    table_text = "\n".join(
        f"| {item['name']} | {item['application']} | {item['proficiency']} |"
        for item in technologies
    )

    return (
        f"# {name}\n\n"
        f"学号：{sid}\n\n"
        f"身份：{role}\n\n"
        f"> {motto}\n\n"
        f"## 技能或爱好\n\n"
        f"{hobby_text}\n\n"
        f"## 本学期最想掌握的技术\n\n"
        f"| 技术名 | 应用场景 | 掌握程度 |\n"
        f"| --- | --- | --- |\n"
        f"{table_text}\n"
    )
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[introduce_self],
  
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "帮我生成一份自我介绍,我的名字是黄鑫远，学号0254250214，身份学生，座右铭，慢功出烂货，欲速则一坨，技能 变大，变小，变大小，掌握请况糟糕，爱好游戏生活运动就按照这样生成"}
    ]
})

print(response["messages"][-1].content)

你的自我介绍已经生成好了！Markdown 内容如下：

---

# 黄鑫远

学号：0254250214

身份：学生

> 慢功出烂货，欲速则一坨

## 技能或爱好

- 游戏
- 生活
- 运动

## 本学期最想掌握的技术

| 技术名 | 应用场景 | 掌握程度 |
| --- | --- | --- |
| 变大 | 够不着高处时假装自己很高 | 糟糕 |
| 变小 | 找角落躲起来摸鱼时 | 糟糕 |
| 变大小 | 随心所欲切换形态（理想中） | 糟糕 |

---

说明一下，由于你的"技能"（变大、变小、变大小）没有提供具体应用场景，我就结合你给的掌握程度"糟糕"帮你脑补了几个好玩的场景。如果你有自己真实的想法，告诉我，我可以帮你改成更正经（或更不正经）的版本 😄


In [13]:
from langchain_community.document_loaders import JSONLoader

#加载json
json_loader=JSONLoader(
    file_path="../asset/load/03-load.json",
    jq_schema=".",
    text_content=False
)
docs=json_loader.load()
print(docs)

[Document(metadata={'source': 'D:\\pycharmproject\\LangChain1\\LangChain2\\asset\\load\\03-load.json', 'seq_num': 1}, page_content='{"messages": [{"sender": "Alice", "content": "Hello, how are you today?", "timestamp": "2023-05-15T10:00:00"}, {"sender": "Bob", "content": "I\'m doing well, thanks for asking!", "timestamp": "2023-05-15T10:02:00"}, {"sender": "Alice", "content": "Would you like to meet for lunch?", "timestamp": "2023-05-15T10:05:00"}, {"sender": "Bob", "content": "Sure, that sounds great!", "timestamp": "2023-05-15T10:07:00"}], "conversation_id": "conv_12345", "participants": ["Alice", "Bob"]}')]


In [11]:
import os
import time
import requests
from dotenv import load_dotenv

from pathlib import Path
env_path = Path(r"D:\pycharmproject\LangChain1\.env")
load_dotenv(dotenv_path=env_path, override=True)




def upload_files(file_paths: list[str]) -> str:
    """批量上传文件"""
    url = "https://mineru.net/api/v4/file-urls/batch"
    api_token = os.getenv("MINERU_API_TOKEN")
    header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_token}",
    }

    files_info = [
        {
            "name": os.path.basename(file_path),
            "is_ocr": True,
            "data_id": f"file_{i}",
        }
        for i, file_path in enumerate(file_paths)
    ]

    data = {
        "enable_formula": True,
        "enable_table": True,
        "language": "ch",
        "files": files_info,
    }

    try:
        response = requests.post(url, headers=header, json=data)
        if response.status_code == 200:
            result = response.json()
            print("response success. result:{}".format(result))

            if result["code"] == 0:
                batch_id = result["data"]["batch_id"]
                urls = result["data"]["file_urls"]
                print("batch_id:{}\nurls:{}".format(batch_id, urls))

                for i in range(0, len(urls)):
                    with open(file_paths[i], "rb") as f:
                        res_upload = requests.put(urls[i], data=f)
                        if res_upload.status_code == 200:
                            print(f"{urls[i]} upload success")
                        else:
                            print(f"{urls[i]} upload failed")
                            return None

                return batch_id
            else:
                print("apply upload url failed, reason:{}".format(result.get("msg")))
                return None
        else:
            print(
                "response not success. status:{} ,result:{}".format(
                    response.status_code, response.text
                )
            )
            return None

    except Exception as err:
        print(err)
        return None


def download_files(batch_id):
    """批量获取任务结果"""
    if not batch_id:
        print("batch_id为空，跳过下载")
        return

    os.makedirs("parsed_files", exist_ok=True)

    url = f"https://mineru.net/api/v4/extract-results/batch/{batch_id}"
    api_token = os.getenv("MINERU_API_TOKEN")
    header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_token}",
    }

    failed_files = set()
    done_files = set()

    while True:
        res = requests.get(url, headers=header)
        result_json = res.json()

        if res.status_code != 200 or result_json.get("code") != 0:
            print("get result failed:", result_json)
            break

        extract_results = result_json["data"]["extract_result"]

        for result in extract_results:
            data_id = result["data_id"]

            if result["state"] == "failed":
                failed_files.add(data_id)

            elif result["state"] == "done" and data_id not in done_files:
                done_files.add(data_id)

                full_zip_url = result["full_zip_url"]
                res_download = requests.get(full_zip_url, stream=True)

                with open(
                    f"parsed_files/{result['file_name']}_{result['data_id']}.zip", "wb"
                ) as f:
                    for chunk in res_download.iter_content(chunk_size=1024):
                        if chunk:
                            f.write(chunk)

        if len(failed_files) + len(done_files) == len(extract_results):
            break

        time.sleep(5)

    for i in failed_files:
        print("failed:", i)

    for i in done_files:
        print("done:", i)


file_paths = ["../asset/load/04-sample.pdf"]
batch_id = upload_files(file_paths)

if batch_id:
    download_files(batch_id)

response not success. status:401 ,result:{"traceId":"4ad75691011f","msgCode":"A0202","msg":"user authenticate failed","data":null,"success":false,"total":0}


In [13]:
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

loader=UnstructuredWordDocumentLoader(
    file_path="../asset/load/05-sgg_chat.docx",
    model="single"
)
docs=loader.load()
print(docs)
print(len(docs))

[Document(metadata={'source': '../asset/load/05-sgg_chat.docx'}, page_content='尚硅谷师资团队评估报告\n\n一、机构基本信息\n\n名称：尚硅谷教育科技有限公司\n总部地址：北京市海淀区中关村软件园创新大厦B座5层\n成立日期：2013年3月15日\n法定代表人：张明哲\n注册资本：人民币2000万元\n员工规模：全职讲师150人，技术助教300人\n官方电话：010-66669999\n官方邮箱：hr@atguigu.com\n\n二、师资团队核心优势\n\n技术背景深厚\n\n90%讲师来自BAT、字节跳动等一线互联网企业，平均技术从业年限8年+\n\n35%讲师拥有开源项目贡献经历（如Apache、Linux基金会项目）\n\n教学能力突出\n\n所有讲师均通过“教学能力三级认证体系”（技术能力、课程设计、课堂表达）\n\n学员平均评分4.9/5.0（2023年内部调研数据）\n\n行业影响力显著\n\n12位讲师担任Oracle/Red Hat/AWS认证官方考官\n\n出版技术书籍28本，累计销量超50万册\n\n三、师资结构分析\n\n类别 人数 占比 典型代表 架构师级讲师 25 16.7% 前阿里P9分布式系统专家 高级研发讲师 80 53.3% 腾讯T4级后台开发工程师 新兴领域讲师 45 30% 原字节AIGC项目组核心成员\n\n教学方向覆盖：\n\n后端开发（Java/Go/Python）\n\n大数据（Hadoop/Spark/Flink）\n\n人工智能（机器学习/深度学习）\n\n云原生（K8s/Service Mesh）\n\n四、教研成果\n\n课程体系\n\n累计开发课程1200+课时，年更新率40%\n\n15门课程入选工信部“紧缺人才培养计划”\n\n学员成就\n\n2023年学员平均就业薪资18K/月（高于行业均值32%）\n\n30名学员获Github年度Star贡献者\n\n企业合作\n\n与200+企业建立人才输送通道（含华为、蚂蚁集团等）\n\n五、潜在风险提示\n\n技术迭代压力\n\nAI编程工具普及对传统教学内容的冲击\n\n需持续增加AIGC、量子计算等前沿领域师资\n\n区域发展不均衡\n\n80%优质讲师集中在北京/深圳校区\

In [15]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

#加载markDown文件
loader=UnstructuredMarkdownLoader(
    file_path="../asset/load/06-load.md",
    model="single",
    strategy="fast"
)
print(loader.load())


[Document(metadata={'source': '../asset/load/06-load.md'}, page_content='自然语言处理技术文档\n\n本文档用于测试UnstructuredMarkdownLoader的中文处理能力。\n\n第一章：简介\n\n自然语言处理(NLP)是人工智能的重要分支，主要技术包括：\n\n文本分类\n\n命名实体识别\n\n机器翻译\n\n情感分析\n\n问答系统\n\n第二章：关键技术\n\n2.1 预训练模型\n\nBERT：双向Transformer编码器\n\nGPT：自回归语言模型\n\nT5：文本到文本转换框架\n\n2.2 代码示例\n\n```python from transformers import pipeline\n\n创建文本分类管道\n\nclassifier = pipeline("text-classification", model="bert-base-chinese")\n\nresult = classifier("这家餐厅的服务很棒！") print(result)')]


In [17]:
from langchain_community.document_loaders import UnstructuredHTMLLoader

#加载html文件
loader=UnstructuredHTMLLoader(
    file_path="../asset/load/07-load.html",
    model="elements",
    strategy="fast"
)
print(loader.load()) 
print(len(loader.load()))

[Document(metadata={'source': '../asset/load/07-load.html'}, page_content='首发于自然语言处理算法与实践\n\nRAG:将检索与生成方式相结合来做生成任务\n\n烛之文\n\n1、前言\n\n在上一篇<kNN-NER：利用knn近邻算法来做命名实体识别>提及到文中提出kNN-NER框架是一种检索式增强的方法（retrieval augmented methods），就去查看有关retrieval augmented的paper，了解其核心思想，觉得检索式增强的方法很适合许多业务场景使用，因其以一种简捷的方式将外部知识融于模型中去。今天就分享一篇来自Facebook AI Research的paper<Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks>，论文提出一种检索式增强生成方法，应用于知识密集型的NLP任务（如问答生成），该篇论文被2020年NeurIPS 会议接收。\n\n文中说到，以BERT之类的大规模预训练模型将很多事实知识信息存入模型中，可以看着是pre-trained parametric类型，尽管以fine-tuned方式在下游任务取得显著的成效，但这类方法仍存在无法精准地获取和操作知识的缺陷。而在上述提及的问题上，传统知识检索的方法能很好的应对，这类方法可以看着是non-parametric memory类型。于是，论文提出检索式增强生成方法（retrieval-augmented generation，RAG），主要思想就是将pre-trained parametric与non-parametric memory结合起来做语言生成任务，将两类模型集成起来提高任务处理效果。\n\n2、RAG方法\n\n上图为论文提出RAG模型的整体示意图。主要包括两大模块：一个检索器（Retriever， p_\\\\eta(z|x) ） + 一个生成器（Generator， p_\\\\theta(y_i|x,z,y_{1:i-1}) ）。前者包括query encoder和document index，分别负责query的编码和文档的索引；后者是一个seq2seq的生成模型。在检索中，使用的是最大内积搜索的方

In [19]:
from langchain_community.document_loaders import PythonLoader
from langchain_community.document_loaders import DirectoryLoader
#批量加载文件
directory_loader=DirectoryLoader(
    path="../asset/load",
    glob="*py",
    use_multithreading=True,
    show_progress=True,
    loader_cls=PythonLoader
)
docs=directory_loader.load()
print(docs)


100%|██████████| 4/4 [00:00<00:00, 1332.05it/s]

[Document(metadata={'source': '..\\asset\\load\\08-fun.py'}, page_content='"""\n一 函数入门\n"""\n# 1.不使用函数\n# 打印欢迎信息1\nprint("********************************")\nprint("*                              *")\nprint("*     欢迎来到Python世界       *")\nprint("*                              *")\nprint("********************************")\n\n# 打印欢迎信息2\nprint("********************************")\nprint("*                              *")\nprint("*     欢迎来到Python世界       *")\nprint("*                              *")\nprint("********************************")\n\n# 打印欢迎信息3\nprint("********************************")\nprint("*                              *")\nprint("*     欢迎来到Python世界       *")\nprint("*                              *")\nprint("********************************")\n\n# 2.使用函数\ndef print_welcome():\n    """打印欢迎信息"""\n    print("********************************")\n    print("*                              *")\n    print("*     欢迎来到Python世界       *")\n    print("*                              *")